In [2]:
import wrds
import pandas as pd
import numpy as np

# 1. 连接 WRDS 数据库
print("Connecting to WRDS...")
db = wrds.Connection()

Connecting to WRDS...
Loading library list...
Done


In [3]:
# 设置时间范围 (例如提取过去20年的数据作为训练集)
start_date = '2004-01-01'
end_date = '2023-12-31'

# ==========================================
# 步骤 1: 提取 CRSP 月度量价数据 (crsp.msf & crsp.msenames)
# ==========================================
print("Extracting CRSP Monthly Data...")
crsp_query = f"""
    SELECT 
        a.permno, a.date, a.ret, a.retx, a.shrout, a.prc, a.vol,
        b.exchcd, b.shrcd, b.siccd
    FROM 
        crsp.msf AS a
    LEFT JOIN 
        crsp.msenames AS b
    ON 
        a.permno = b.permno
        AND b.namedt <= a.date
        AND a.date <= b.nameendt
    WHERE 
        a.date BETWEEN '{start_date}' AND '{end_date}'
        AND b.shrcd IN (10, 11) -- 仅保留普通股 (剔除 ADR, REITs 等)
        AND b.exchcd IN (1, 2, 3) -- 仅保留 NYSE, AMEX, NASDAQ (剔除场外交易)
"""
crsp = db.raw_sql(crsp_query)

Extracting CRSP Monthly Data...


In [4]:
crsp.head()

,permno,date,ret,retx,shrout,prc,vol,exchcd,shrcd,siccd
0,10001,2004-01-30,0.010084,0.010084,2596.0,6.01,2050.0,3,11,4920
1,10001,2004-02-27,0.079867,0.079867,2596.0,6.49,1077.0,3,11,4920
2,10001,2004-03-31,0.117103,0.117103,2598.0,7.25,1350.0,3,11,4920
3,10001,2004-04-30,-0.012414,-0.012414,2598.0,7.16,695.0,3,11,4920
4,10001,2004-05-28,-0.079609,-0.079609,2598.0,6.59,622.0,3,11,4920


In [5]:

# 数据清洗：处理退市收益率和计算市值 (Size)
crsp['date'] = pd.to_datetime(crsp['date'])
crsp['prc'] = np.abs(crsp['prc']) # 处理价格为负的情况（CRSP中负价格表示没有成交时的买卖中间价）
crsp['me'] = crsp['prc'] * crsp['shrout'] # 计算市值 (Market Equity)


In [6]:


# ==========================================
# 步骤 2: 提取 Compustat 年度财务数据 (comp.funda)
# ==========================================
print("Extracting Compustat Annual Fundamentals...")
# 这里仅提取几个核心变量作为演示：总资产(at), 账面所有者权益(ceq), 净收入(ni)
comp_query = f"""
    SELECT 
        gvkey, datadate, fyear, 
        at, ceq, ni, revt, cogs, xint, xsga
    FROM 
        comp.funda
    WHERE 
        datadate BETWEEN '{start_date}' AND '{end_date}'
        AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C' 
"""
comp = db.raw_sql(comp_query)
comp['datadate'] = pd.to_datetime(comp['datadate'])


Extracting Compustat Annual Fundamentals...


In [7]:
# ==========================================
# 步骤 3: 提取 CCM 链接表 (crsp.ccmxpf_linktable) 
# ==========================================
print("Extracting CRSP/Compustat Linktable...")
ccm_query = """
    SELECT 
        gvkey, lpermno AS permno, linktype, linkprim, linkdt, linkenddt
    FROM 
        crsp.ccmxpf_linktable
    WHERE 
        linktype IN ('LU', 'LC') -- 仅保留高质量链接
        AND linkprim IN ('P', 'C') -- 保留主要证券
"""
ccm = db.raw_sql(ccm_query)
ccm['linkdt'] = pd.to_datetime(ccm['linkdt'])
ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt']).fillna(pd.to_datetime('today'))


Extracting CRSP/Compustat Linktable...


In [8]:

# ==========================================
# 步骤 4: 执行合并 (Merge CRSP and Compustat)
# ==========================================
print("Merging Datasets...")
# 将财务数据通过 GVKEY 匹配到链接表，获得对应的 PERMNO
comp = pd.merge(comp, ccm, on=['gvkey'], how='inner')

# 确保财务数据的发布时间在链接有效时间段内
comp = comp[(comp['datadate'] >= comp['linkdt']) & (comp['datadate'] <= comp['linkenddt'])]

# 为了避免前视偏差 (Look-ahead bias)，通常将年度财务数据滞后 6 个月 (假设年报在次年6月底前全部公布完毕)
comp['merge_date'] = comp['datadate'] + pd.DateOffset(months=6)
comp['merge_year'] = comp['merge_date'].dt.year
comp['merge_month'] = comp['merge_date'].dt.month

crsp['merge_year'] = crsp['date'].dt.year
crsp['merge_month'] = crsp['date'].dt.month

Merging Datasets...


In [9]:





# 最终合并：将滞后处理后的财务数据与每月的 CRSP 量价数据对齐
final_df = pd.merge(crsp, comp, on=['permno', 'merge_year', 'merge_month'], how='left')

# 前向填充财务数据 (因为财报是一年一次，而量价是月度的，使得每个月都有最新的基本面特征)
final_df = final_df.sort_values(['permno', 'date'])
final_df = final_df.groupby('permno').ffill()

print("Data Extraction Complete. Shape:", final_df.shape)
db.close()

Data Extraction Complete. Shape: (973177, 27)


In [10]:
final_df.head()

,date,ret,retx,shrout,prc,vol,exchcd,shrcd,siccd,me,...,ni,revt,cogs,xint,xsga,linktype,linkprim,linkdt,linkenddt,merge_date
0,2004-01-30,0.010084,0.010084,2596.0,6.01,2050.0,3,11,4920,15601.96,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT
1,2004-02-27,0.079867,0.079867,2596.0,6.49,1077.0,3,11,4920,16848.04,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT
2,2004-03-31,0.117103,0.117103,2598.0,7.25,1350.0,3,11,4920,18835.5,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT
3,2004-04-30,-0.012414,-0.012414,2598.0,7.16,695.0,3,11,4920,18601.68,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT
4,2004-05-28,-0.079609,-0.079609,2598.0,6.59,622.0,3,11,4920,17120.82,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,NaT,NaT,NaT


In [11]:
import pandas as pd
import numpy as np

# 假设 df 是上一段程序输出的 final_df
df = final_df.copy()

# 确保数据按股票和时间正确排序，这对于计算动量和滞后极其重要
df = df.sort_values(['permno', 'date'])

# ---------------------------------------------------------
# 1. 规模特征 (Size)
# ---------------------------------------------------------
# 习惯上取对数处理极值
df['Size'] = np.log(df['me'])

# ---------------------------------------------------------
# 2. 价值特征 (Value - Book-to-Market)
# ---------------------------------------------------------
# 账面所有者权益 (ceq) / 市值 (me)
df['BM'] = df['ceq'] / df['me']
# 处理可能出现的负账面价值或极值
df['BM'] = np.where(df['BM'] < 0, np.nan, df['BM']) 

# ---------------------------------------------------------
# 3. 盈利能力特征 (Profitability - Operating Profitability)
# ---------------------------------------------------------
# 营业利润 = 收入(revt) - 销货成本(cogs) - 销售及管理费用(xsga) - 利息支出(xint)
# 缩放变量通常使用账面权益(ceq)或总资产(at)
df['OP'] = (df['revt'] - df['cogs'].fillna(0) - df['xsga'].fillna(0) - df['xint'].fillna(0)) / df['ceq']

# ---------------------------------------------------------
# 4. 投资特征 (Investment - Asset Growth)
# ---------------------------------------------------------
# 总资产增长率：(当期总资产 - 上期总资产) / 上期总资产
df['at_lag1'] = df.groupby('permno')['at'].shift(1)
df['INV'] = (df['at'] - df['at_lag1']) / df['at_lag1']

# ---------------------------------------------------------
# 5. 短期反转特征 (Short-term Reversal - MOM1m)
# ---------------------------------------------------------
# 过去1个月的收益率 (t-1)
df['MOM1m'] = df.groupby('permno')['ret'].shift(1)

# ---------------------------------------------------------
# 6. 动量特征 (Momentum - MOM12m)
# ---------------------------------------------------------
# 经典定义：过去12个月的累计收益率，但跳过最近1个月 (即 t-12 到 t-2)
# 先计算对数收益率以便于连加计算累计收益
df['log_ret'] = np.log(1 + df['ret'])
# 滚动求和11个月（t-12 到 t-2）
df['MOM12m'] = df.groupby('permno')['log_ret'].apply(
    lambda x: x.shift(2).rolling(window=11).sum()
).reset_index(level=0, drop=True)
# 转回普通收益率
df['MOM12m'] = np.exp(df['MOM12m']) - 1

# ---------------------------------------------------------
# 截面标准化 (Cross-sectional Rank Normalization) - 极度重要！
# ---------------------------------------------------------
# 神经网络对输入特征的尺度非常敏感。在横截面定价中，我们通常将每个月的特征映射到 [-1, 1] 区间
features = ['Size', 'BM', 'OP', 'INV', 'MOM1m', 'MOM12m']

def cs_normalize(group):
    # 使用 Rank 转化可以完美消除财务数据中的极端异常值（Outliers）
    return (group.rank() - 1) / (len(group) - 1) * 2 - 1

for feat in features:
    df[f'{feat}_norm'] = df.groupby('date')[feat].transform(cs_normalize)

KeyError: 'permno'

In [ ]:
import wrds
import pandas as pd
import numpy as np
import datetime
from pandas_datareader import data as pdr
from scipy.stats.mstats import winsorize
import warnings
warnings.filterwarnings('ignore')

class AssetPricingDataPipeline:
    def __init__(self, start_date='1980-01-01', end_date='2023-12-31'):
        self.start_date = start_date
        self.end_date = end_date
        self.df = None
        self.macro_df = None
        self.feature_cols = []
        
    def download_wrds_data(self):
        """
        核心步骤 1: 从 WRDS 下载并合并 CRSP 与 Compustat 数据
        """
        print("Connecting to WRDS and downloading micro data...")
        db = wrds.Connection()
        
        # 提取 CRSP 月度量价
        crsp_query = f"""
            SELECT a.permno, a.date, a.ret, a.prc, a.shrout, a.vol, b.exchcd, b.shrcd
            FROM crsp.msf AS a
            LEFT JOIN crsp.msenames AS b
            ON a.permno = b.permno AND b.namedt <= a.date AND a.date <= b.nameendt
            WHERE a.date BETWEEN '{self.start_date}' AND '{self.end_date}'
            AND b.shrcd IN (10, 11) AND b.exchcd IN (1, 2, 3)
        """
        crsp = db.raw_sql(crsp_query)
        crsp['date'] = pd.to_datetime(crsp['date']) + pd.offsets.MonthEnd(0)
        crsp['me'] = np.abs(crsp['prc']) * crsp['shrout']
        
        # 提取 Compustat 财务数据
        comp_query = f"""
            SELECT gvkey, datadate, at, ceq, ni, revt, cogs, xsga, xint
            FROM comp.funda
            WHERE datadate BETWEEN '{self.start_date}' AND '{self.end_date}'
            AND indfmt='INDL' AND datafmt='STD' AND popsrc='D' AND consol='C'
        """
        comp = db.raw_sql(comp_query)
        comp['datadate'] = pd.to_datetime(comp['datadate'])
        
        # 提取 CCM 链接表并合并
        ccm_query = """
            SELECT gvkey, lpermno AS permno, linkdt, linkenddt
            FROM crsp.ccmxpf_linktable
            WHERE linktype IN ('LU', 'LC') AND linkprim IN ('P', 'C')
        """
        ccm = db.raw_sql(ccm_query)
        ccm['linkdt'] = pd.to_datetime(ccm['linkdt'])
        ccm['linkenddt'] = pd.to_datetime(ccm['linkenddt']).fillna(pd.to_datetime('today'))
        
        comp = pd.merge(comp, ccm, on='gvkey', how='inner')
        comp = comp[(comp['datadate'] >= comp['linkdt']) & (comp['datadate'] <= comp['linkenddt'])]
        
        # 财务数据滞后 6 个月避免前视偏差
        comp['merge_date'] = comp['datadate'] + pd.DateOffset(months=6) + pd.offsets.MonthEnd(0)
        
        # 按照年月合并 CRSP 和 Compustat
        crsp['year_month'] = crsp['date'].dt.to_period('M')
        comp['year_month'] = comp['merge_date'].dt.to_period('M')
        
        self.df = pd.merge(crsp, comp, on=['permno', 'year_month'], how='left')
        self.df = self.df.sort_values(['permno', 'date_x']).reset_index(drop=True)
        self.df.rename(columns={'date_x': 'date'}, inplace=True)
        db.close()
        print("WRDS Data downloaded and merged.")

    def download_macro_data(self):
        """
        核心步骤 2: 从 FRED 下载宏观状态变量 (TERM, DEF, VIX)
        """
        print("Fetching Macro Predictors from FRED...")
        # VIXCLS 是 VIX 指数，T10Y3M 是期限利差，BAAA 和 AAA 用于算信用利差
        tickers = ['T10Y3M', 'BAAA', 'AAA', 'VIXCLS']
        # 多拉取一点历史数据以防滞后计算产生 NaN
        start = pd.to_datetime(self.start_date) - pd.DateOffset(years=1)
        macro_raw = pdr.get_data_fred(tickers, start, self.end_date)
        
        # 转换为月度末数据
        macro_df = macro_raw.resample('M').last().reset_index()
        macro_df.rename(columns={'DATE': 'date'}, inplace=True)
        macro_df['date'] = macro_df['date'] + pd.offsets.MonthEnd(0)
        
        # 计算特征
        macro_df['TERM'] = macro_df['T10Y3M']
        macro_df['DEF'] = macro_df['BAAA'] - macro_df['AAA']
        macro_df['VIX'] = macro_df['VIXCLS']
        
        # 极其重要：作为 t 期的条件预测变量，宏观数据无需再滞后（因为月末已经能够观测到当月的数据），
        # 但如果是用 t 月预测 t+1 月，特征向量直接用当期月末值即可。
        self.macro_df = macro_df[['date', 'TERM', 'DEF', 'VIX']].ffill().dropna()
        print("Macro Data Ready.")

    def build_features_and_target(self):
        """
        核心步骤 3: 截面特征构建、目标收益率波动率缩放与截面标准化
        """
        print("Building Micro Features, Tensor Products, and Scaling Targets...")
        df = self.df.copy()
        
        # --- 1. 构建底层截面特征 ---
        df['Size'] = np.log(df['me'])
        df['BM'] = df['ceq'] / df['me']
        df['BM'] = np.where(df['BM'] < 0, np.nan, df['BM'])
        df['OP'] = (df['revt'] - df['cogs'].fillna(0) - df['xsga'].fillna(0) - df['xint'].fillna(0)) / df['ceq']
        
        df['at_lag1'] = df.groupby('permno')['at'].shift(1)
        df['INV'] = (df['at'] - df['at_lag1']) / df['at_lag1']
        df['MOM1m'] = df.groupby('permno')['ret'].shift(1)
        
        df['log_ret'] = np.log(1 + df['ret'])
        df['MOM12m'] = df.groupby('permno')['log_ret'].apply(
            lambda x: x.shift(2).rolling(window=11).sum()
        ).reset_index(level=0, drop=True)
        df['MOM12m'] = np.exp(df['MOM12m']) - 1
        
        # 合并宏观数据
        df = pd.merge(df, self.macro_df, on='date', how='inner')
        
        # --- 2. 目标收益率构建 (Target Normalization) - 顶级防御！ ---
        # 目标是预测下一期的收益率 t+1
        df['target_ret_raw'] = df.groupby('permno')['ret'].shift(-1)
        
        # 波动率缩放 (Volatility Scaling)：使用当期 VIX 缩放下一期收益率，防止 MDN 方差崩溃
        # VIX 是年化百分比，除以 100 转化为小数，除以 sqrt(12) 转化为月度波动率代理
        df['target_ret_scaled'] = df['target_ret_raw'] / (df['VIX'] / 100 / np.sqrt(12))
        
        # 截面缩尾 (Cross-sectional Winsorization 1%)：避免极少数妖股产生的极大/极小目标值摧毁 NLL 梯度
        def cs_winsorize(group):
            # 只有在样本量足够时才缩尾
            if len(group.dropna()) > 50:
                return winsorize(group, limits=[0.01, 0.01])
            return group
        
        df['target_ret_final'] = df.groupby('date')['target_ret_scaled'].transform(cs_winsorize)
        
        # --- 3. 特征张量化与截面秩标准化 (Cross-sectional Rank Normalization) ---
        micro_feats = ['Size', 'BM', 'OP', 'INV', 'MOM1m', 'MOM12m']
        macro_feats = ['TERM', 'DEF'] # VIX 主要用于目标缩放，也可作为特征
        
        tensor_cols = []
        for micro in micro_feats:
            # 截面秩标准化 [-1, 1]
            norm_col = f'{micro}_norm'
            df[norm_col] = df.groupby('date')[micro].transform(
                lambda x: (x.rank() - 1) / (len(x.dropna()) - 1) * 2 - 1
            ).fillna(0) # 缺失值用中位数0填充
            tensor_cols.append(norm_col)
            
            # 张量积交互
            for macro in macro_feats:
                interact_name = f'{norm_col}_x_{macro}'
                df[interact_name] = df[norm_col] * df[macro]
                tensor_cols.append(interact_name)
                
        self.feature_cols = tensor_cols
        
        # 清理最终数据集（剔除缺失目标变量的行，即最后一期）
        self.df = df.dropna(subset=['target_ret_final'] + self.feature_cols)
        print(f"Data pipeline complete. Tensor features count: {len(self.feature_cols)}")

    def expanding_window_generator(self, initial_train_years=20, val_years=2, test_years=1):
        """
        核心步骤 4: 纯样本外扩展窗口生成器 (Expanding Window)
        解决审稿人对于“未来信息泄露”的致命质疑。
        """
        dates = np.sort(self.df['date'].unique())
        
        initial_train_months = initial_train_years * 12
        val_months = val_years * 12
        test_months = test_years * 12
        
        start_idx = 0
        current_split_idx = initial_train_months
        
        while current_split_idx + val_months + test_months <= len(dates):
            # Expanding Train: 始终从最开始训练，窗口不断扩大
            train_dates = dates[start_idx : current_split_idx]
            val_dates = dates[current_split_idx : current_split_idx + val_months]
            test_dates = dates[current_split_idx + val_months : current_split_idx + val_months + test_months]
            
            train_data = self.df[self.df['date'].isin(train_dates)]
            val_data = self.df[self.df['date'].isin(val_dates)]
            test_data = self.df[self.df['date'].isin(test_dates)]
            
            window_info = {
                'train': (train_dates[0].strftime('%Y-%m'), train_dates[-1].strftime('%Y-%m')),
                'val': (val_dates[0].strftime('%Y-%m'), val_dates[-1].strftime('%Y-%m')),
                'test': (test_dates[0].strftime('%Y-%m'), test_dates[-1].strftime('%Y-%m'))
            }
            
            yield window_info, train_data, val_data, test_data
            
            # 窗口向前推移 test_months (例如每年重新估计一次)
            current_split_idx += test_months

# ==========================================
# 执行流 (Execution)
# ==========================================
if __name__ == "__main__":
    # 初始化 Pipeline (例如跑 1990-2023 的数据)
    pipeline = AssetPricingDataPipeline(start_date='1990-01-01', end_date='2023-12-31')
    
    # 依次执行模块 (如果您本地已经有提取好的 WRDS 数据，可以直接 load，跳过 download_wrds_data)
    pipeline.download_wrds_data()
    pipeline.download_macro_data()
    pipeline.build_features_and_target()
    
    # 初始化扩展窗口 (初始训练 20年, 验证 2年, 测试 1年)
    generator = pipeline.expanding_window_generator(initial_train_years=20, val_years=2, test_years=1)
    
    print("\n--- Expanding Window Architecture Initiated ---")
    for window_idx, (info, train_df, val_df, test_df) in enumerate(generator):
        print(f"Window {window_idx + 1} | Train [{info['train'][0]} to {info['train'][1]}] -> Val [{info['val'][0]} to {info['val'][1]}] -> Test [{info['test'][0]} to {info['test'][1]}]")
        
        # 准备 PyTorch 张量
        X_train = train_df[pipeline.feature_cols].values
        Y_train = train_df['target_ret_final'].values
        
        # 确保提前退出测试（实际跑模型时去掉这个 break）
        if window_idx == 2:
            break

In [1]:
import pandas as pd
import sqlite3
import urllib.request
import zipfile
import os

def download_open_asset_pricing_features(db_path='tail_risk_data.db'):
    print("=== Downloading Open Asset Pricing (Chen & Zimmermann) Dataset ===")
    
    # Chen & Zimmermann 公开数据集的长期稳定下载链接 (包含 200+ 个特征的 CSV 压缩包)
    url = "https://github.com/OpenAssetPricing/Data/raw/main/Firm_Level_Characteristics/signed_predictors_dl.zip"
    zip_path = "signed_predictors_dl.zip"
    extract_folder = "oap_data"
    
    if not os.path.exists(zip_path):
        print("Downloading zip file (~300MB)... This may take a few minutes.")
        urllib.request.urlretrieve(url, zip_path)
    
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_folder)
    
    # 寻找到解压出来的 CSV 文件
    csv_filename = [f for f in os.listdir(extract_folder) if f.endswith('.csv')][0]
    csv_path = os.path.join(extract_folder, csv_filename)
    
    print("Reading and parsing features...")
    # 该数据集通常按 permno 和 yyyymm 组织
    df_features = pd.read_csv(csv_path)
    
    # 格式化日期，确保与你的微观数据能完美 merge
    df_features['yyyymm'] = df_features['yyyymm'].astype(str)
    df_features['date'] = pd.to_datetime(df_features['yyyymm'], format='%Y%m') + pd.offsets.MonthEnd(0)
    
    # 剔除无用的列
    df_features.drop(columns=['yyyymm'], errors='ignore', inplace=True)
    
    print(f"Feature processing complete. Shape: {df_features.shape}")
    print(f"Total anomalies available: {len(df_features.columns) - 2}") # 减去 permno 和 date
    
    # 存入你的 SQLite 数据库
    print("Saving to SQLite database as 'oap_features'...")
    conn = sqlite3.connect(db_path)
    df_features.to_sql('oap_features', conn, if_exists='replace', index=False)
    
    # 建立索引以极大加快阶段一的读取速度
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_oap_date ON oap_features (date);")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_oap_permno ON oap_features (permno);")
    conn.close()
    
    print("=== OAP Pipeline Complete! ===")

# 使用示例
download_open_asset_pricing_features()

=== Downloading Open Asset Pricing (Chen & Zimmermann) Dataset ===


HTTPError: HTTP Error 404: Not Found

In [6]:
import openassetpricing as oap
import sqlite3
import pandas as pd

def download_open_asset_pricing_features_v3(db_path='tail_risk_data.db'):
    print("=== Downloading OAP Dataset via Official Python API (v3) ===")
    
    try:
        openap = oap.OpenAP()
        print("Successfully connected to Open Asset Pricing servers.")
    except Exception as e:
        print(f"Connection failed: {e}")
        return
    
    # 核心修正：使用我们刚刚侦测出来的正确 API 接口
    print("Downloading 200+ firm characteristics (This will take a few minutes, ~1.6GB)...")
    try:
        # 调用 dl_all_signals，通常默认返回 pandas dataframe
        df_features = openap.dl_all_signals('pandas')
    except Exception as e:
        print(f"Download error: {e}")
        return

    print(f"Data downloaded successfully! Shape: {df_features.shape}")
    
    # 清洗列名，确保能和你的微观数据完美合并
    if 'yyyymm' in df_features.columns:
        df_features['yyyymm'] = df_features['yyyymm'].astype(str)
        df_features['date'] = pd.to_datetime(df_features['yyyymm'], format='%Y%m') + pd.offsets.MonthEnd(0)
        df_features.drop(columns=['yyyymm'], inplace=True)
    elif 'Date' in df_features.columns:
        df_features.rename(columns={'Date': 'date'}, inplace=True)
        
    if 'PERMNO' in df_features.columns:
        df_features.rename(columns={'PERMNO': 'permno'}, inplace=True)
        
    # 将缺失值填充或者处理掉 (特征表里通常有大量的 NaN，这里我们先保留存入数据库)
    print("Saving features to SQLite database as 'oap_features'...")
    conn = sqlite3.connect(db_path)
    df_features.to_sql('oap_features', conn, if_exists='replace', index=False, chunksize=10000)
    
    cursor = conn.cursor()
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_oap_date ON oap_features (date);")
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_oap_permno ON oap_features (permno);")
    conn.close()
    
    print("=== OAP Pipeline Complete! You now possess the academic Holy Grail. ===")

# 运行它！
download_open_asset_pricing_features_v3()

=== Downloading OAP Dataset via Official Python API (v3) ===
Successfully connected to Open Asset Pricing servers.
Loading library list...
Done

Data is downloaded: 11 mins
Data downloaded successfully! Shape: (5416424, 214)
Saving features to SQLite database as 'oap_features'...
=== OAP Pipeline Complete! You now possess the academic Holy Grail. ===


In [ ]:
# read oap_features from SQLite to verify
conn = sqlite3.connect('tail_risk_data.db')
df_oap = pd.read_sql_query("SELECT * FROM oap_features LIMIT 20;", conn)
conn.close()   
df_oap

In [9]:
df_oap

,permno,AM,AOP,AbnormalAccruals,Accruals,AccrualsBM,Activism1,Activism2,AdExp,AgeIPO,...,skew1,std_turn,tang,zerotrade12M,zerotrade1M,zerotrade6M,Price,Size,STreversal,date
0,10000,None,None,None,None,None,None,None,None,None,...,None,None,None,None,NaN,None,-1.475907,-2.778819,0.000000,1986-01-31 00:00:00
1,10000,None,None,None,None,None,None,None,None,None,...,None,None,None,None,4.785175e-08,None,-1.178655,-2.481568,0.257143,1986-02-28 00:00:00
2,10000,None,None,None,None,None,None,None,None,None,...,None,None,None,None,1.023392e-07,None,-1.490091,-2.793004,-0.365385,1986-03-31 00:00:00
3,10000,None,None,None,None,None,None,None,None,None,...,None,None,None,None,7.467463e-08,None,-1.386294,-2.719452,0.098592,1986-04-30 00:00:00
4,10000,None,None,None,None,None,None,None,None,None,...,None,None,None,None,7.649551e-08,None,-1.134423,-2.467581,0.222656,1986-05-31 00:00:00


Tables in the database: [('micro_data',), ('macro_data',), ('mdn_predictions',), ('final_pricing_factors',), ('oap_features',)]
